# QTEMP v1.0.0 - final analytical disposition

This notebook closes QTEMP as an immutable analytical implementation and feature-selection result. No QTEMP feature enters the validated primary feature set; four deterministic outputs remain available for explicitly exploratory/descriptive or monitoring use, and splice remains dropped.


In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import Image, Markdown, display

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "config" / "project.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing config/project.yaml")

PROJECT_ROOT = find_project_root(Path.cwd())
FINALIZER = PROJECT_ROOT / "scripts" / "finalize_qtemp_analytical_disposition.py"
FINAL_ROOT = PROJECT_ROOT / "MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates" / "temporal_discontinuity" / "qtemp-v1.0.0-analytical-final-no-retained"
assert FINALIZER.exists(), FINALIZER
print("Project root:", PROJECT_ROOT)
print("Final analytical archive:", FINAL_ROOT)


## 1. Create or verify the final immutable analytical archive


In [ ]:
subprocess.run([sys.executable, str(FINALIZER), "--project-root", str(PROJECT_ROOT)], check=True)
status = json.loads((FINAL_ROOT / "manifests" / "qtemp_v100_final_status.json").read_text(encoding="utf-8"))
display(status)


## 2. Final feature-level decisions


In [ ]:
decisions = pd.read_csv(FINAL_ROOT / "validation" / "qtemp_v100_feature_decisions.csv")
display(decisions)
assert decisions["primary_validated_set"].astype(str).str.lower().eq("false").all()


## 3. Final G1-G10 disposition


In [ ]:
gates = pd.read_csv(FINAL_ROOT / "validation" / "qtemp_v100_gate_summary.csv")
display(gates)
assert gates.loc[gates.gate.eq("G9"), "state"].iloc[0] == "N/A_NO_RETAINED_PRIMARY_EVENT_FEATURES"
assert gates.loc[gates.gate.eq("G10"), "state"].iloc[0] == "FINALIZED_NO_RETAINED_PRIMARY_FEATURES"


## 4. Empirical extent retained for descriptive reporting


In [ ]:
features = pd.read_csv(FINAL_ROOT / "tables" / "qtemp_v100_exploratory_features.csv")
rows = []
for feature in [
    "qtemp_dropout_duration_fraction", "qtemp_dropout_event_rate_per_min",
    "qtemp_frozen_audio_duration_fraction", "qtemp_frozen_audio_event_rate_per_min",
]:
    rows.append({
        "feature": feature,
        "recordings": len(features),
        "available": int(features[feature].notna().sum()),
        "zero": int(features[feature].eq(0).sum()),
        "positive": int(features[feature].gt(0).sum()),
        "final_role": "exploratory/monitoring; not primary validated",
    })
display(pd.DataFrame(rows))


## 5. Final standardized validation gallery A-J


In [ ]:
for letter, slug in [
    ("A", "construct-response"), ("B", "discriminant-specificity"),
    ("C", "transformation-contract"), ("D", "support-availability"),
    ("E", "parameter-sensitivity"), ("F", "empirical-distribution"),
    ("G", "signal-examples"), ("H", "reliability-redundancy"),
    ("I", "event-verification"), ("J", "ml-handoff"),
]:
    display(Markdown(f"### Panel {letter}"))
    path = FINAL_ROOT / "figures" / f"panel-{letter}_{slug}" / f"qtemp_v100_panel-{letter}_{slug}.png"
    display(Image(filename=str(path), width=1150))


## 6. Full checklist and immutable manifest


In [ ]:
checklist = pd.read_csv(FINAL_ROOT / "validation" / "qtemp_v100_validation_checklist.csv")
manifest = pd.read_csv(FINAL_ROOT / "manifests" / "qtemp_v100_final_artifact_sha256.csv")
display(checklist)
display(manifest)
print("Checklist rows:", len(checklist))
print("Manifested artifacts:", len(manifest))


## 7. Final manuscript-safe conclusion


In [ ]:
print("QTEMP FINALIZED: analytical implementation frozen; zero retained primary validated features.")
print("Exploratory evidence: 2/519 dropout-positive recordings; 0/519 frozen-audio-positive recordings.")
print("Event verification is N/A because no QTEMP event feature is retained in the validated primary set.")
print("Splice remains dropped. QTEMP must not be described as a validated packet-loss or physiological biomarker family.")
